# Re-pack the full-dataset groups A, B, C into a few large parts (for molab)

Kaggle unpacked our uploaded `.tar` files into ~15,000 small files per dataset, and then refuses to serve the whole
dataset as one download (404) and rate-limits file-by-file downloads (429). This notebook packs each group again as
**4 GB parts named `kitti_full_<g>.part000, .part001, ...`** (Kaggle does not unpack those) and uploads them as new
private datasets `<you>/semantickitti-mra-full-<g>-parts`. molab then downloads ~4 files per group and joins them.

**Before running**
1. Right panel: **Add Input** -> Your Work -> Datasets -> add `semantickitti-mra-full-a`, `-b` and `-c`. Internet **On**.
   Accelerator: none needed (CPU is fine).
2. Add-ons -> Secrets: `KAGGLE_API_TOKEN` switched on for this notebook (same as the build notebook).
3. **Save Version -> Save & Run All (Commit)**. About 20-30 min per group. Groups already uploaded are skipped.

In [ ]:
KAGGLE_USER = "daniyalimran"
GROUPS = ["a", "b", "c"]
PART_SIZE = "4000M"

import os, glob, json, shutil, subprocess, time
subprocess.run("pip install -q -U kaggle", shell=True, check=True)
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
def _secret(n):
    try:
        return _s.get_secret(n)
    except Exception:
        return None
if _secret("KAGGLE_API_TOKEN"):
    os.environ["KAGGLE_API_TOKEN"] = _secret("KAGGLE_API_TOKEN")
else:
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = _secret("KAGGLE_USERNAME"), _secret("KAGGLE_KEY")
r = subprocess.run(["kaggle", "datasets", "list", "--mine"], capture_output=True, text=True)
assert r.returncode == 0, "Kaggle credentials rejected: " + r.stdout + r.stderr
print("OK: Kaggle CLI works")

In [ ]:
def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

def find_input(g):
    """The mounted input folder of semantickitti-mra-full-<g> that contains kitti_full/."""
    for root, dirs, _ in os.walk("/kaggle/input"):
        if "kitti_full" in dirs and f"semantickitti-mra-full-{g}" in root:
            return root
        if root.count("/") > 6:
            dirs[:] = []
    raise SystemExit(f"input semantickitti-mra-full-{g} not found under /kaggle/input: add it with 'Add Input'")

def exists_on_kaggle(ref):
    r = subprocess.run(["kaggle", "datasets", "status", ref], capture_output=True, text=True)
    return r.returncode == 0 and "ready" in r.stdout.lower()

for g in GROUPS:
    ref = f"{KAGGLE_USER}/semantickitti-mra-full-{g}-parts"
    if exists_on_kaggle(ref):
        log(f"{ref} already exists - skipped"); continue
    src = find_input(g)
    n = sum(len(f) for _, _, f in os.walk(f"{src}/kitti_full"))
    log(f"group {g}: {src}/kitti_full ({n} files) -> {ref} | free disk {shutil.disk_usage('/tmp').free/1e9:.0f} GB")
    up = f"/tmp/repack_{g}"; shutil.rmtree(up, ignore_errors=True); os.makedirs(up)
    # tar the folder (kitti_full/...) and cut the stream into numbered parts that Kaggle will not unpack
    subprocess.run(f"tar -C '{src}' -cf - kitti_full | split -b {PART_SIZE} -d -a 3 - '{up}/kitti_full_{g}.part'",
                   shell=True, check=True)
    parts = sorted(os.listdir(up))
    log(f"parts: {parts} ({sum(os.path.getsize(f'{up}/{p}') for p in parts)/1e9:.1f} GB)")
    json.dump({"title": f"SemanticKITTI MRA full {g.upper()} parts", "id": ref, "licenses": [{"name": "other"}],
               "description": "Group " + g.upper() + " of kaggle/build_full_dataset.ipynb packed as one tar split into "
                              "parts: cat kitti_full_" + g + ".part* | tar xf -  (molab/prepare_data.py does this)."},
              open(f"{up}/dataset-metadata.json", "w"), indent=1)
    log(f"uploading {ref} (private)")
    subprocess.run(["kaggle", "datasets", "create", "-p", up, "-q"], check=True)
    shutil.rmtree(up, ignore_errors=True)
    log(f"UPLOADED {ref}")
log("done")